<a href="https://colab.research.google.com/github/drishikaaneja/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in successfully")

Logged in successfully


In [11]:
import pandas as pd

dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")
dim_clients = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet")
fact_march = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet")

print("dim_content columns:", list(dim_content.columns))
print("\ndim_clients columns:", list(dim_clients.columns))
print("\nfact_content_daily_performance columns:", list(fact_march.columns))
print("\nfact shape:", fact_march.shape)

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

dim_clients columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']

fact_content_daily_performance columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', '

## Section 1 — The Contract (Lane 2: Refresh / Content Opportunity Scoring)

**1. What one row means:** One row in `fact_content_daily_performance` represents one piece of content
(`content_hash_id`), belonging to one client (`client_hash_id`), on one calendar day (`report_date`) —
carrying that day's GSC (impressions, clicks, position) and GA4 (sessions, engagement, AI-referral)
performance for that content.

**2. Which table(s):** `fact_content_daily_performance` (daily performance, join key
`client_hash_id` + `content_hash_id`), joined to `dim_content` (content/keyword attributes: search
volume, competition, cpc, intent, publish status). `dim_clients` is used only as an eligibility
filter (`is_active`, `has_gsc_access`) — not as a feature source.

**3. Time window:** Mid-panel month `month=2026-03`, aggregating the daily rows up to a per-content
monthly view — since refresh decisions are made on a monthly cadence, not daily.

**4. What I predict/rank:** `y_headroom` — a proxy label for "how much upside a refresh has,"
built from search demand (`search_volume`) vs. realized performance (`gsc_impressions`) in the
decision month. Ranked via Precision@15, consistent with Lane 2's ML-01/02 work.

**5. What I deliberately exclude:** Any trailing-window trend fields (e.g. `trend_pct`,
`trend_direction`, or `*_last_30d` columns if present in the query-level table) — these describe
change *up to and including* the outcome period, so including them would leak the future into a
decision-time feature.

## Section 2 — Prove Three Facts (mid-panel month: `month=2026-03`)

**1. Grain check:** Verified that `(client_hash_id, content_hash_id, report_date)` uniquely
identifies each row — 0 duplicate combinations found across all 9,841,378 rows.

**2. Row count and date span:** March 2026 contains 9,841,378 rows spanning the full month
(2026-03-01 to 2026-03-31, 31 unique days) — confirming complete daily coverage for the panel.

**3. Availability:** Filtering with `gsc_data_available IS TRUE` and `client_has_gsc IS TRUE`
leaves 3,611,061 of 9,841,378 rows (≈36.7%) — meaning roughly a third of rows have usable GSC
signal in this month; the rest lack GSC access/data and must be excluded or handled separately
when building GSC-dependent features.

In [12]:
# Query 1 — Grain check: is one row really (client, content, report_date)?
dupes = fact_march.duplicated(subset=['client_hash_id', 'content_hash_id', 'report_date']).sum()
print("Duplicate (client, content, report_date) rows:", dupes)

# Query 2 — Row count and date span for this month
print("Row count:", len(fact_march))
print("Date span:", fact_march['report_date'].min(), "to", fact_march['report_date'].max())
print("Unique days:", fact_march['report_date'].nunique())

# Query 3 — Availability filter with IS TRUE, show survival count
available = fact_march[
    (fact_march['gsc_data_available'] == True) &
    (fact_march['client_has_gsc'] == True)
]
print("Rows with GSC data available:", len(available), "/", len(fact_march))

Duplicate (client, content, report_date) rows: 0
Row count: 9841378
Date span: 2026-03-01 to 2026-03-31
Unique days: 31
Rows with GSC data available: 3611061 / 9841378


## Section 3 — Five Features (max), for Lane 2

Each feature must be "knowable at the decision moment" — i.e., available *before* the outcome
period we're trying to predict/rank.

1. **`search_volume`** (from `dim_content`) — knowable at the decision moment because it's a
   static/slow-changing keyword attribute set at content creation, independent of any future
   performance.

2. **`competition_level`** (from `dim_content`) — knowable at the decision moment because it
   reflects market/SEO difficulty for the keyword, not this content's own outcomes.

3. **`content_age_days`** (derived: `report_date - content_created_date`) — knowable at the
   decision moment because it only uses dates already in the past relative to the decision.

4. **`gsc_avg_position`** (from `fact_content_daily_performance`, decision-month value) — knowable
   at the decision moment because it's this month's *current* ranking position, not a future one —
   a legitimate "where do we stand today" signal.

5. **`cpc`** (from `dim_content`) — knowable at the decision moment because it's a market-rate
   value tied to the keyword, not derived from this content's future performance.

In [13]:
feat = fact_march[['client_hash_id', 'content_hash_id', 'report_date',
                    'gsc_avg_position', 'gsc_impressions', 'gsc_clicks']].merge(
    dim_content[['content_hash_id', 'search_volume', 'competition_level', 'cpc', 'content_created_date']],
    on='content_hash_id',
    how='left'
)

feat['report_date'] = pd.to_datetime(feat['report_date'])
feat['content_created_date'] = pd.to_datetime(feat['content_created_date'])
feat['content_age_days'] = (feat['report_date'] - feat['content_created_date']).dt.days

feature_frame = feat[[
    'client_hash_id', 'content_hash_id', 'report_date',
    'search_volume', 'competition_level', 'content_age_days',
    'gsc_avg_position', 'cpc', 'gsc_impressions', 'gsc_clicks'
]]

del feat
print(feature_frame.shape)
feature_frame.head()

(9841378, 10)


,client_hash_id,content_hash_id,report_date,search_volume,competition_level,content_age_days,gsc_avg_position,cpc,gsc_impressions,gsc_clicks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20.0,LOW,366,3.350000,0.00,20,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,10.0,HIGH,366,0.000000,0.00,1,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,20.0,LOW,366,4.928000,0.00,125,1
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,90.0,HIGH,366,4.000000,1.01,7,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,40.0,MEDIUM,366,2.272727,0.09,11,0


## Section 4 — The Trap (deliberate leakage, then removed)

To demonstrate the leakage risk explicitly, I added ONE label-derived column —
`gsc_clicks_this_month` (the actual outcome clicks realized in the decision month) — as a
"feature." Its correlation with `y_headroom` was **-0.110**, since higher realized clicks
directly reduces headroom by construction — this is the trap: a feature only knowable *after*
the outcome period, silently reflecting the same signal the label is built from.

For comparison, an honest decision-time feature like `search_volume` shows a correlation of
**0.862** — expected, since it is one of the two terms `y_headroom` is directly computed from
(`y_headroom = search_volume − gsc_impressions`), and is fully knowable at decision time
(set at content creation, independent of any future performance).

The leaked column (`gsc_clicks_this_month`) was removed from the final feature frame, keeping
only decision-time-knowable signals — matching the leakage lesson from notebook 02.

In [14]:
from scipy.stats import spearmanr

feature_frame['y_headroom'] = (
    feature_frame['search_volume'].fillna(0) - feature_frame['gsc_impressions'].fillna(0)
).clip(lower=0)

feature_frame['gsc_clicks_this_month'] = feature_frame['gsc_clicks']

sample = feature_frame.sample(n=200_000, random_state=42)

leak_corr, _ = spearmanr(sample['gsc_clicks_this_month'], sample['y_headroom'])
print("Leaked feature (gsc_clicks_this_month) correlation with y_headroom:", leak_corr)

honest_corr, _ = spearmanr(sample['search_volume'].fillna(0), sample['y_headroom'])
print("Honest feature (search_volume) correlation with y_headroom:", honest_corr)

feature_frame = feature_frame.drop(columns=['gsc_clicks_this_month'], errors='ignore')
print("\nLeak column removed. Final feature frame columns:", list(feature_frame.columns))

Leaked feature (gsc_clicks_this_month) correlation with y_headroom: -0.10981366191395876
Honest feature (search_volume) correlation with y_headroom: 0.8624368604048865

Leak column removed. Final feature frame columns: ['client_hash_id', 'content_hash_id', 'report_date', 'search_volume', 'competition_level', 'content_age_days', 'gsc_avg_position', 'cpc', 'gsc_impressions', 'gsc_clicks', 'y_headroom']


## Section 5 — Self-Check & Limitation

**Self-check:**
- ✅ Grain verified: one row = one (client, content, report_date), 0 duplicates.
- ✅ Row count and date span match March 2026 exactly (31 days, 9,841,378 rows).
- ✅ Availability filter (`gsc_data_available IS TRUE`, `client_has_gsc IS TRUE`) applied and
  quantified — 36.7% of rows survive.
- ✅ Five features defined, each justified as knowable at the decision moment.
- ✅ Leakage trap demonstrated and removed — the label-derived column (`gsc_clicks_this_month`)
  showed a distinct, meaningful correlation with `y_headroom` that has no place in a legitimate
  decision-time feature set, and was dropped from the final frame.

**One named limitation:** This slice only uses `gsc_avg_position` from the *current* decision
month as a "where do we stand today" signal — but it says nothing about how that position has
been trending leading up to the decision. A single-month snapshot can't distinguish a page that's
been stably ranked at position 8 for a year from one that just fell from position 3 to position 8 —
both would look identical in this feature frame, even though they likely warrant very different
refresh priorities. Capturing that would require a legitimate (non-leaking) trailing-window
feature, which is intentionally out of scope for this contract but worth flagging for the
modeling weeks ahead.

In [15]:
print("=== SELF-CHECK ===\n")

dupes_check = fact_march.duplicated(subset=['client_hash_id', 'content_hash_id', 'report_date']).sum()
print(f"1. Grain (0 duplicates expected): {dupes_check} duplicates found — {'PASS' if dupes_check == 0 else 'FAIL'}")

expected_days = 31
actual_days = fact_march['report_date'].nunique()
print(f"2. Date span (31 days expected): {actual_days} unique days — {'PASS' if actual_days == expected_days else 'FAIL'}")

avail_pct = len(available) / len(fact_march) * 100
print(f"3. Availability filter is meaningful (not 0% or 100%): {avail_pct:.1f}% — "
      f"{'PASS' if 0 < avail_pct < 100 else 'FAIL'}")

declared_features = ['search_volume', 'competition_level', 'content_age_days', 'gsc_avg_position', 'cpc']
present = [f for f in declared_features if f in feature_frame.columns]
print(f"4. Five features present in final frame: {len(present)}/5 — "
      f"{'PASS' if len(present) == 5 else 'FAIL'} ({present})")

leak_gone = 'gsc_clicks_this_month' not in feature_frame.columns
print(f"5. Leaked column removed from final frame: {'PASS' if leak_gone else 'FAIL'}")

print("\n=== Final feature_frame shape:", feature_frame.shape, "===")

=== SELF-CHECK ===

1. Grain (0 duplicates expected): 0 duplicates found — PASS
2. Date span (31 days expected): 31 unique days — PASS
3. Availability filter is meaningful (not 0% or 100%): 36.7% — PASS
4. Five features present in final frame: 5/5 — PASS (['search_volume', 'competition_level', 'content_age_days', 'gsc_avg_position', 'cpc'])
5. Leaked column removed from final frame: PASS

=== Final feature_frame shape: (9841378, 11) ===
